# RAG Pipeline — Notebook Report
Domain: **Tech / Office Equipment Manuals**

This notebook builds and evaluates the full RAG pipeline: load → chunk → embed → store → retrieve → generate → evaluate.

In [ ]:
!pip install -q pypdf sentence-transformers chromadb ollama pandas

## 2.1 Load & Inspect
How many documents/pages? What formats? Which files failed to parse or need OCR?

In [ ]:
import os
from pypdf import PdfReader

DATA_DIR = "../data/raw_corpus"  # put your source PDFs here

docs = []
for fname in os.listdir(DATA_DIR):
    if fname.lower().endswith(".pdf"):
        path = os.path.join(DATA_DIR, fname)
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        docs.append({"filename": fname, "num_pages": len(reader.pages), "text": text, "chars": len(text)})

print(f"Loaded {len(docs)} documents")
for d in docs:
    status = "OK" if d["chars"] > 50 else "NEEDS OCR / FAILED"
    print(f"- {d['filename']}: {d['num_pages']} pages, {d['chars']} chars -> {status}")

**Findings (fill in after running):** X documents, Y total pages, format = PDF. Files that failed to parse or need OCR: ...

## 2.2 Chunking Strategy
Split documents into chunks and justify the chosen chunk size / overlap.

In [ ]:
CHUNK_SIZE = 500     # characters
CHUNK_OVERLAP = 50   # characters

def chunk_text(text, source, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append({"text": chunk, "source": source})
        start += chunk_size - overlap
    return chunks

all_chunks = []
for d in docs:
    all_chunks.extend(chunk_text(d["text"], d["filename"]))

print(f"Total chunks: {len(all_chunks)}")

**Justification (fill in):** Chose fixed-size chunking with 500 chars / 50 overlap because ... (explain trade-off: too small loses context, too large dilutes retrieval precision).

## 2.3 Embeddings & Vector Store
Generate embeddings, store in Chroma, persist to disk.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
VECTOR_STORE_DIR = "../backend/data/vector_store"

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
collection = client.get_or_create_collection("rag_chunks")

texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True).tolist()
ids = [f"chunk_{i}" for i in range(len(all_chunks))]
metadatas = [{"source": c["source"]} for c in all_chunks]

collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
print("Vector store persisted to", VECTOR_STORE_DIR)

## 2.4 Retrieval & Prompting
Implement retrieval, test against 10 sample questions, build the grounded prompt template.

In [ ]:
def retrieve(question, top_k=4):
    q_emb = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=top_k)
    return [
        {"text": doc, "source": meta["source"]}
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]

PROMPT_TEMPLATE = """Answer using ONLY the context below. Cite the source in brackets.

Context:
{context}

Question: {question}
Answer:"""

def build_prompt(question, chunks):
    context = "\n\n".join(f"[source: {c['source']}]\n{c['text']}" for c in chunks)
    return PROMPT_TEMPLATE.format(context=context, question=question)

sample_questions = [
    "Question 1 here", "Question 2 here", "Question 3 here", "Question 4 here",
    "Question 5 here", "Question 6 here", "Question 7 here", "Question 8 here",
    "Question 9 here", "Question 10 here",
]

for q in sample_questions:
    chunks = retrieve(q)
    print(q, "->", [c["source"] for c in chunks])

## 2.5 Vision Component (Extended Track only)
Run inference with a pretrained YOLOv8 model (no training needed) on the image dataset, and decide how detections feed into the RAG prompt context.

In [ ]:
# !pip install -q ultralytics
# from ultralytics import YOLO
# model = YOLO("yolov8n.pt")  # pretrained on COCO, no training needed
# results = model("../data/images/sample.jpg")
# detected_labels = [model.names[int(box.cls)] for box in results[0].boxes]
# print(detected_labels)  # e.g. ['laptop', 'keyboard']
# -> feed detected_labels into the RAG prompt as extra context, e.g.
#    "The uploaded image shows: laptop, keyboard. " + question

## 2.6 Evaluation
Results for at least 10 test questions: was context relevant? was the answer grounded or hallucinated?

In [ ]:
import pandas as pd

# TODO: after calling the LLM (via ollama) for each sample question, fill this table
eval_results = pd.DataFrame([
    {"question": "...", "retrieved_source": "...", "answer": "...", "correct": True},
])
eval_results

**Failure analysis (fill in):** Main failure cases observed were ... Mitigations tried: ...

## 2.7 Export
Vector store is already persisted above to `../backend/data/vector_store`, ready for the backend to load without rebuilding. Record config used:

In [ ]:
import json
config = {"embedding_model": EMBEDDING_MODEL_NAME, "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP}
with open(f"{VECTOR_STORE_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)
print(config)